# Single-Checkpoint Robustness

**Purpose**: evaluate one saved checkpoint at a time using a clean, repeatable robustness workflow aligned to the Run10 test window.

### What This Notebook Does
1. Sync repo, restore saved training/results zip from Drive, and rebuild the evaluation config from metadata.
2. Run deterministic horizon tests for `1yr`, `2yr`, `3yr`, and `4yr`.
3. Run stratified stochastic robustness for the same horizons (`20` runs per horizon).
4. Run unconstrained Monte Carlo start-date robustness for the same horizons (`20` runs per horizon).
5. Compare deterministic results against `SPY` and equal-weight benchmarks.
6. Produce asset-level allocation/opportunity-set visuals and export all artifacts.

**Run10-aligned test start**: `2022-01-03`
**Checkpoint under test**: set `CHECKPOINT_TAG` in the setup cell.


---
## 1) Connect to Colab VM and sync repository

In [56]:
import os, subprocess, sys

EVAL_REPO_DIR = "/content/tcn_tape_vectorized_version_clean"
EVAL_BRANCH = "main"  # override if the checkpoint/config lives on another branch
EVAL_GH_REPO = "Dave-DKings/tcn_tape_vectorized_version"

if not os.path.exists(EVAL_REPO_DIR):
    clone_cmd = ["git", "clone"]
    if EVAL_BRANCH:
        clone_cmd += ["-b", EVAL_BRANCH]
    clone_cmd += [f"https://github.com/{EVAL_GH_REPO}.git", EVAL_REPO_DIR]
    subprocess.run(clone_cmd, check=True)
else:
    subprocess.run(["git", "-C", EVAL_REPO_DIR, "fetch", "origin"], check=False)
    if EVAL_BRANCH:
        subprocess.run(["git", "-C", EVAL_REPO_DIR, "checkout", EVAL_BRANCH], check=False)
        subprocess.run(["git", "-C", EVAL_REPO_DIR, "pull", "--ff-only", "origin", EVAL_BRANCH], check=False)
    else:
        subprocess.run(["git", "-C", EVAL_REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(EVAL_REPO_DIR)
print("[OK] Repo synced")
print("   repo:", EVAL_REPO_DIR)
print("   branch:", EVAL_BRANCH if EVAL_BRANCH else "current default")


[OK] Repo synced
   repo: /content/tcn_tape_vectorized_version_clean
   branch: feature/run6-tuning-20260306


In [ ]:
RUN_ID = "run10"
CHECKPOINT_TAG = "exp6_tape_hw_ep00004_shp1p259"  # set the checkpoint you want to evaluate
NOTEBOOK_TAG = "single_checkpoint_robustness"


In [58]:
# Quick GPU check (Colab/Jupyter)
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow version:", tf.__version__)
print("GPUs found:", len(gpus))

if gpus:
    print("GPU active: YES")
    for i, g in enumerate(gpus):
        print(f"  [{i}] {g}")
    print("Current device:", tf.test.gpu_device_name())
else:
    print("GPU active: NO")

TensorFlow version: 2.19.0
GPUs found: 1
GPU active: YES
  [0] PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Current device: /device:GPU:0


In [60]:
# Install project requirements in Colab VM
import subprocess, sys
from pathlib import Path

REPO_DIR = Path(globals().get("EVAL_REPO_DIR", "/content/tcn_tape_vectorized_version_clean"))
REQ_FILE = REPO_DIR / "requirements.txt"

if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

print("Using python:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", str(REQ_FILE)], check=True)

print("[OK] Requirements installed. If key libs were upgraded, restart runtime before continuing.")

Using python: /usr/bin/python3
[OK] Requirements installed. If key libs were upgraded, restart runtime before continuing.


## 2) Mount Drive and restore saved results zip

In [ ]:
from pathlib import Path
import zipfile

EVAL_RESTORE_FROM_ZIP = True
EVAL_RESTORE_DIR = Path("/content/eval_restore")
EVAL_RESTORE_ZIP_PATH_OVERRIDE = None

if EVAL_RESTORE_FROM_ZIP:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    drive_root = Path("/content/drive/MyDrive")

    candidate_paths = []
    if EVAL_RESTORE_ZIP_PATH_OVERRIDE:
        candidate_paths.append(Path(EVAL_RESTORE_ZIP_PATH_OVERRIDE))
    candidate_paths += [
        drive_root / f"tcn_tape_vectorized_{RUN_ID}.zip",
        drive_root / f"tcn_tape_vectorized_version_{RUN_ID}.zip",
        drive_root / "tcn_tape_vectorized_version_clean.zip",
    ]
    if drive_root.exists():
        candidate_paths += sorted(drive_root.glob(f"*{RUN_ID}*.zip"), reverse=True)
        candidate_paths += sorted(drive_root.glob("*tcn*tape*.zip"), reverse=True)

    zip_path = next((p for p in candidate_paths if p.exists()), None)
    if zip_path is None:
        raise FileNotFoundError(
            f"No restore zip found on Drive. Checked override={EVAL_RESTORE_ZIP_PATH_OVERRIDE} and run-id patterns for {RUN_ID}."
        )

    print("[DOC] Restore zip:", zip_path)
    EVAL_RESTORE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(EVAL_RESTORE_DIR)
    print("[OK] Restored results from zip into", EVAL_RESTORE_DIR)
else:
    print("EVAL_RESTORE_FROM_ZIP=False")


In [62]:
from pathlib import Path
candidates = [
    EVAL_RESTORE_DIR / "tcn_fusion_results",
    Path(EVAL_REPO_DIR) / "tcn_fusion_results",
]
EVAL_RESULTS_ROOT = next((p for p in candidates if p.exists()), candidates[0])
print("[OK] EVAL_RESULTS_ROOT =", EVAL_RESULTS_ROOT)
print("   actor ckpts:", len(list(EVAL_RESULTS_ROOT.rglob("*_actor.weights.h5"))))

[OK] EVAL_RESULTS_ROOT = /content/eval_restore/tcn_fusion_results
   actor ckpts: 101


## 3) TF precision + imports

In [63]:
import tensorflow as tf
tf.keras.mixed_precision.set_global_policy("float32")
print(tf.keras.mixed_precision.global_policy())

<DTypePolicy "float32">


In [64]:
import copy, json, re, shutil, time
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

from src.config import get_active_config, apply_run5_overrides
from src.data_utils import DataProcessor
from src.notebook_helpers.tcn_phase1 import (
    prepare_phase1_dataset, create_experiment6_result_stub,
    evaluate_experiment6_checkpoint, load_training_metadata_into_config,
    build_evaluation_track_summary, Phase1Dataset,
    split_dataset_by_date, identify_covariance_columns,
)
print("[OK] All imports loaded")

[OK] All imports loaded


## 4) Evaluation run settings

In [ ]:
import re

EVAL_RANDOM_SEED = 42
EVAL_FORCE_TEST_START_DATE = "2022-01-03"
EVAL_ALLOW_RESUME = True

# Deterministic horizons: one pass each from the Run10 test start.
EVAL_DET_HORIZONS = {
    "1yr": 252,
    "2yr": 504,
    "3yr": 756,
    "4yr": 1008,
}

# Stochastic horizons: same horizon family, two sampling schemes.
EVAL_STOCHASTIC_HORIZONS = dict(EVAL_DET_HORIZONS)
EVAL_STRATIFIED_RUNS_PER_HORIZON = 20
EVAL_STRATIFIED_STRATA_PER_HORIZON = 10
EVAL_MONTE_CARLO_RUNS_PER_HORIZON = 20

EVAL_DETERMINISTIC_MODE = "mean"
EVAL_STOCHASTIC_MODE = "sample"
EVAL_METADATA_PATH_OVERRIDE = None

EVAL_ASSET_UNIVERSE = [
    "MSFT", "GOOGL", "JPM", "JNJ", "XOM",
    "PG", "NEE", "LIN", "CAT", "UNH",
]

EVAL_ENSURE_DRIVE_MOUNT_FOR_OUTPUT = True
if EVAL_ENSURE_DRIVE_MOUNT_FOR_OUTPUT and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"[WARN] Could not mount Drive for outputs: {e}")

EVAL_SAVE_LOGS = False
EVAL_SAVE_ARTIFACTS = False

ARTIFACT_TAG = re.sub(r"[^A-Za-z0-9._-]+", "_", CHECKPOINT_TAG).strip("_") if CHECKPOINT_TAG else "set_checkpoint_tag"
DRIVE_OUTPUT_DIR = Path(f"/content/drive/MyDrive/{NOTEBOOK_TAG}_{RUN_ID}_{ARTIFACT_TAG}")
LOCAL_OUTPUT_DIR = Path(f"/content/{NOTEBOOK_TAG}_{ARTIFACT_TAG}")
OUTPUT_DIR = DRIVE_OUTPUT_DIR if Path("/content/drive/MyDrive").exists() else LOCAL_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("[OK] Settings configured")
print(f"   Checkpoint: {CHECKPOINT_TAG}")
print(f"   Artifact tag: {ARTIFACT_TAG}")
print(f"   Run10 test start: {EVAL_FORCE_TEST_START_DATE}")
print(f"   Deterministic horizons: {EVAL_DET_HORIZONS}")
print(f"   Stratified stochastic: {EVAL_STRATIFIED_RUNS_PER_HORIZON} runs/horizon across {EVAL_STRATIFIED_STRATA_PER_HORIZON} strata")
print(f"   Monte Carlo stochastic: {EVAL_MONTE_CARLO_RUNS_PER_HORIZON} runs/horizon")
print(f"   Resume enabled: {EVAL_ALLOW_RESUME}")
print(f"   Output dir: {OUTPUT_DIR}")
print(f"   Output persistence: {'Drive' if OUTPUT_DIR == DRIVE_OUTPUT_DIR else 'Local'}")


## 5) Locate normalized CSV

In [66]:
target_dir = Path(EVAL_REPO_DIR) / "data"
target_dir.mkdir(parents=True, exist_ok=True)
target_path = target_dir / "master_features_NORMALIZED.csv"

candidates = []
candidates += list(EVAL_RESTORE_DIR.rglob("master_features_NORMALIZED.csv"))
candidates += list(EVAL_RESTORE_DIR.rglob("*normalized*.csv"))
seen, ordered = set(), []
for p in candidates:
    if p.suffix.lower() == ".csv" and str(p.resolve()) not in seen:
        seen.add(str(p.resolve())); ordered.append(p)
if not ordered:
    raise FileNotFoundError(f"No normalized CSV found under {EVAL_RESTORE_DIR}")
best = max(ordered, key=lambda p: p.stat().st_mtime)
shutil.copy2(best, target_path)
print("[OK] Normalized CSV:", best, "->", target_path)

[OK] Normalized CSV: /content/eval_restore/data_exports/phase1_prep_20260309_172322_test_normalized.csv -> /content/tcn_tape_vectorized_version_clean/data/master_features_NORMALIZED.csv


## 6) Build eval config from training metadata + feature lock

In [ ]:
def eval_extract_trained_state_layout(metadata_dict):
    arch = metadata_dict.get("Architecture_Settings", {}) or {}
    effective = arch.get("agent_params_effective", {}) or {}
    template = arch.get("agent_params_template", {}) or {}
    layout = effective.get("state_layout") or template.get("state_layout")
    if not isinstance(layout, dict):
        raise ValueError("state_layout not found")
    active_cols = layout.get("active_feature_columns")
    if not isinstance(active_cols, list) or not active_cols:
        raise ValueError("active_feature_columns missing")
    return layout, list(dict.fromkeys(active_cols))


def eval_apply_metadata_feature_lock(cfg, trained_cols):
    probe_cfg = copy.deepcopy(cfg)
    probe_cfg.setdefault("feature_params", {}).setdefault("feature_selection", {})
    probe_cfg["feature_params"]["feature_selection"]["disable_features"] = False
    probe_cfg["feature_params"]["feature_selection"]["disabled_features"] = []
    core = list(dict.fromkeys(DataProcessor(probe_cfg).get_feature_columns("phase1")))
    for c in trained_cols:
        if c not in core:
            core.append(c)
    gap = {c for c in core if c not in set(trained_cols)}
    existing = set(cfg.get("feature_params", {}).get("feature_selection", {}).get("disabled_features", []))
    disabled = sorted(existing.union(gap))
    fs = cfg.setdefault("feature_params", {}).setdefault("feature_selection", {})
    fs["disable_features"] = True
    fs["disabled_features"] = disabled
    return core, disabled


def eval_bind_trained_feature_layout(processor, trained_cols):
    cols = list(dict.fromkeys(trained_cols))
    base = processor.get_feature_columns

    def _locked(phase="phase1"):
        return list(cols) if str(phase).lower() == "phase1" else base(phase)

    processor.get_feature_columns = _locked
    return processor


eval_config = copy.deepcopy(get_active_config("phase1"))
eval_logs_dir = EVAL_RESULTS_ROOT / "logs"
meta_files = sorted(eval_logs_dir.glob("*_metadata.json"), key=lambda p: p.stat().st_mtime, reverse=True)

meta_override = globals().get("EVAL_METADATA_PATH_OVERRIDE", None)
if meta_override:
    EVAL_METADATA_PATH = Path(meta_override)
    if not EVAL_METADATA_PATH.exists():
        raise FileNotFoundError(f"EVAL_METADATA_PATH_OVERRIDE not found: {EVAL_METADATA_PATH}")
else:
    if not meta_files:
        raise FileNotFoundError(f"No metadata files under: {eval_logs_dir}")
    EVAL_METADATA_PATH = meta_files[0]

print("[DOC] Metadata:", EVAL_METADATA_PATH)
with open(EVAL_METADATA_PATH, "r", encoding="utf-8") as f:
    eval_metadata = json.load(f)

eval_config = load_training_metadata_into_config(EVAL_METADATA_PATH, copy.deepcopy(eval_config), verbose=True)

# Run10-aligned evaluation dataset split.
if EVAL_FORCE_TEST_START_DATE:
    ts = pd.to_datetime(EVAL_FORCE_TEST_START_DATE)
    eval_config["TRAIN_TEST_SPLIT_DATE"] = (ts - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    eval_config["ANALYSIS_START_DATE"] = "2016-01-01"

# Hard requirements for one-checkpoint Run10 robustness testing.
eval_config.setdefault("feature_params", {}).setdefault("fundamental_features", {})["enabled"] = False
eval_config["feature_params"].setdefault("actuarial_params", {})["enabled"] = False
eval_config.setdefault("training_params", {})["num_parallel_envs"] = 1
eval_config["ASSET_TICKERS"] = list(EVAL_ASSET_UNIVERSE)
eval_config["NUM_ASSETS"] = len(EVAL_ASSET_UNIVERSE)
eval_config["agent_params"]["actor_critic_type"] = "TCN_FUSION"
eval_config["agent_params"]["use_fusion"] = True
eval_config["agent_params"]["use_attention"] = False

ae = (eval_metadata.get("Architecture_Settings", {}) or {}).get("agent_params_effective", {}) or {}
for k, fb in {
    "fusion_cross_asset_mixer_enabled": False,
    "fusion_cross_asset_mixer_layers": 1,
    "fusion_cross_asset_mixer_expansion": 2.0,
    "fusion_cross_asset_mixer_dropout": 0.1,
    "fusion_asset_identity_enabled": True,
    "fusion_context_cross_attention_enabled": False,
    "fusion_context_cross_attention_heads": 4,
    "fusion_context_cross_attention_dropout": 0.1,
    "fusion_per_asset_alpha_head": True,
    "fusion_alpha_head_hidden_dims": [128, 64],
    "fusion_alpha_head_dropout": 0.05,
    "recurrent_memory_enabled": False,
    "regime_conditioning_enabled": True,
    "regime_conditioning_mode": "film",
    "regime_conditioning_hidden_dim": 32,
    "regime_conditioning_dropout": 0.05,
    "state_augmentation_enabled": False,
    "distributional_critic_enabled": True,
    "distributional_num_quantiles": 17,
    "dual_head_enabled": False,
}.items():
    eval_config["agent_params"][k] = ae.get(k, eval_config["agent_params"].get(k, fb))

ppo = eval_config["agent_params"].setdefault("ppo_params", {})
ppo.setdefault("popart_enabled", True)
ppo.setdefault("multi_horizon_reward_enabled", False)
ppo.setdefault("dual_head_consistency_coef", 0.0)

trained_state_layout, trained_active_feature_columns = eval_extract_trained_state_layout(eval_metadata)
eval_config["agent_params"]["state_layout"] = copy.deepcopy(trained_state_layout)
eval_config["agent_params"]["asset_feature_dim"] = int(trained_state_layout.get("asset_feature_dim", 0) or 0)
eval_config["agent_params"]["global_feature_dim"] = int(trained_state_layout.get("global_feature_dim", 0) or 0)
eval_config["agent_params"]["num_assets"] = int(trained_state_layout.get("num_assets", len(EVAL_ASSET_UNIVERSE)) or len(EVAL_ASSET_UNIVERSE))
core_all_cols, eval_disabled = eval_apply_metadata_feature_lock(eval_config, trained_active_feature_columns)
print(f"[OK] Config built | features: {len(trained_active_feature_columns)} | disabled: {len(eval_disabled)}")

EVAL_FORCE_SINGLE_ENV = True
EVAL_APPLY_EXECUTION_TURNOVER_OVERRIDES = True
EVAL_OVERRIDE_ACTION_EXEC_BETA = 0.60
EVAL_OVERRIDE_TURNOVER_PENALTY = float(
    eval_config.get("training_params", {}).get("evaluation_turnover_penalty_scalar", 1.0)
)

if EVAL_FORCE_SINGLE_ENV:
    eval_config.setdefault("training_params", {})["num_parallel_envs"] = 1

if EVAL_APPLY_EXECUTION_TURNOVER_OVERRIDES:
    eval_config.setdefault("training_params", {})["evaluation_action_execution_beta"] = EVAL_OVERRIDE_ACTION_EXEC_BETA
    eval_config.setdefault("training_params", {})["evaluation_turnover_penalty_scalar"] = EVAL_OVERRIDE_TURNOVER_PENALTY
    eval_config.setdefault("environment_params", {})["action_execution_beta"] = EVAL_OVERRIDE_ACTION_EXEC_BETA

print("eval num_parallel_envs:", eval_config.get("training_params", {}).get("num_parallel_envs"))
print("eval action_execution_beta:", eval_config.get("training_params", {}).get("evaluation_action_execution_beta"))
print("eval turnover_penalty_scalar:", eval_config.get("training_params", {}).get("evaluation_turnover_penalty_scalar"))
print("eval env.action_execution_beta:", eval_config.get("environment_params", {}).get("action_execution_beta"))
print("eval TRAIN_TEST_SPLIT_DATE:", eval_config.get("TRAIN_TEST_SPLIT_DATE"))
print("eval memory/regime/distributional:", {
    "recurrent_memory_enabled": eval_config.get("agent_params", {}).get("recurrent_memory_enabled"),
    "regime_conditioning_enabled": eval_config.get("agent_params", {}).get("regime_conditioning_enabled"),
    "state_augmentation_enabled": eval_config.get("agent_params", {}).get("state_augmentation_enabled"),
    "distributional_critic_enabled": eval_config.get("agent_params", {}).get("distributional_critic_enabled"),
    "distributional_num_quantiles": eval_config.get("agent_params", {}).get("distributional_num_quantiles"),
})


## 7) Build evaluation dataset

In [68]:
if "eval_phase1_data" in globals(): del eval_phase1_data

normalized_path = Path(EVAL_REPO_DIR) / "data" / "master_features_NORMALIZED.csv"
master_df_norm = pd.read_csv(normalized_path)
master_df_norm["Date"] = pd.to_datetime(master_df_norm["Date"], utc=True, errors="coerce").dt.tz_localize(None)
master_df_norm = master_df_norm.dropna(subset=["Date"]).sort_values(["Date","Ticker"]).reset_index(drop=True)
analysis_start = pd.to_datetime(eval_config.get("ANALYSIS_START_DATE","2003-09-02"))
analysis_end = pd.to_datetime(eval_config.get("ANALYSIS_END_DATE","2025-09-01"))
master_df_norm = master_df_norm[(master_df_norm["Date"]>=analysis_start)&(master_df_norm["Date"]<=analysis_end)].copy()
master_df_norm = master_df_norm[master_df_norm["Ticker"].isin(EVAL_ASSET_UNIVERSE)].copy()

split_date = eval_config.get("TRAIN_TEST_SPLIT_DATE")
train_df, test_df, train_end_date, test_start_date = split_dataset_by_date(master_df_norm, date_column="Date", split_date=split_date)
eval_processor = DataProcessor(eval_config)
eval_processor = eval_bind_trained_feature_layout(eval_processor, trained_active_feature_columns)
eval_phase1_data = Phase1Dataset(
    master_df=master_df_norm, train_df=train_df, test_df=test_df, scalers={},
    train_end_date=train_end_date, test_start_date=test_start_date,
    covariance_columns=identify_covariance_columns(master_df_norm.columns),
    data_processor=eval_processor)
print(f"[OK] Dataset built | Test: {test_df.shape} | {test_df['Date'].min()} to {test_df['Date'].max()}")

[SPLIT]  TIME-BASED TRAIN/TEST SPLIT (Fixed date: 2019-12-31)
   [WARN]  No training dates available (empty dataset).
   Test:  2020-01-02 => 2025-08-29 (1423 days, 5.6 years, 14230 rows)
[OK] Dataset built | Test: (14230, 66) | 2020-01-02 00:00:00 to 2025-08-29 00:00:00


## 8) Eval Config Snapshot + Healthcheck


In [ ]:
def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def _safe_write_json(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_json_safe(payload), f, indent=2, default=str)


def _select_dict(d, keys):
    return {k: d.get(k) for k in keys if k in d}


phase1_test_df = eval_phase1_data.test_df.copy()
phase1_test_df["Date"] = pd.to_datetime(phase1_test_df["Date"])
test_dates = pd.Series(phase1_test_df["Date"].dropna().unique()).sort_values().reset_index(drop=True)
agent_params = eval_config.get("agent_params", {}) or {}
ppo_params = agent_params.get("ppo_params", {}) or {}
training_params = eval_config.get("training_params", {}) or {}
environment_params = eval_config.get("environment_params", {}) or {}
feature_params = eval_config.get("feature_params", {}) or {}
state_layout = agent_params.get("state_layout", {}) or {}

hyperparameter_snapshot = {
    "run_id": RUN_ID,
    "notebook_tag": NOTEBOOK_TAG,
    "checkpoint_tag": CHECKPOINT_TAG,
    "metadata_path": str(EVAL_METADATA_PATH),
    "training_params": _select_dict(training_params, [
        "num_parallel_envs",
        "evaluation_action_execution_beta",
        "evaluation_turnover_penalty_scalar",
        "deterministic_eval_mode",
        "stochastic_eval_mode",
    ]),
    "environment_params": _select_dict(environment_params, [
        "action_execution_beta",
        "turnover_penalty_scalar",
        "transaction_cost_pct",
        "reward_scaling",
    ]),
    "agent_params": _select_dict(agent_params, [
        "actor_critic_type",
        "num_assets",
        "asset_feature_dim",
        "global_feature_dim",
        "use_fusion",
        "use_attention",
        "fusion_cross_asset_mixer_enabled",
        "recurrent_memory_enabled",
        "regime_conditioning_enabled",
        "regime_conditioning_mode",
        "state_augmentation_enabled",
        "distributional_critic_enabled",
        "distributional_num_quantiles",
        "dual_head_enabled",
        "alpha_activation",
        "logit_temperature",
        "alpha_cap",
    ]),
    "ppo_params": _select_dict(ppo_params, [
        "gamma",
        "gae_lambda",
        "clip_ratio",
        "entropy_coef",
        "target_kl",
        "actor_learning_rate",
        "critic_learning_rate",
        "actor_lr_schedule",
        "critic_lr_schedule",
        "rollout_schedule",
        "batch_size_schedule",
        "temperature_schedule",
        "aux_return_pred_coef",
        "aux_return_pred_coef_schedule",
        "lagrangian_cvar_penalty_scale",
        "tail_aware_advantage_weight",
        "drawdown_penalty_coef",
        "drawdown_target",
        "drawdown_tolerance",
        "hhi_coef",
        "dispersion_coef",
        "target_std",
    ]),
}

eval_healthcheck = {
    "run_id": RUN_ID,
    "notebook_tag": NOTEBOOK_TAG,
    "checkpoint_tag": CHECKPOINT_TAG,
    "artifact_tag": ARTIFACT_TAG,
    "metadata_path": str(EVAL_METADATA_PATH),
    "eval_results_root": str(EVAL_RESULTS_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "eval_force_test_start_date": EVAL_FORCE_TEST_START_DATE,
    "train_test_split_date": eval_config.get("TRAIN_TEST_SPLIT_DATE"),
    "analysis_start_date": eval_config.get("ANALYSIS_START_DATE"),
    "analysis_end_date": eval_config.get("ANALYSIS_END_DATE"),
    "test_date_min": test_dates.min().date().isoformat() if len(test_dates) else None,
    "test_date_max": test_dates.max().date().isoformat() if len(test_dates) else None,
    "test_days": int(len(test_dates)),
    "test_rows": int(len(phase1_test_df)),
    "num_assets": int(len(EVAL_ASSET_UNIVERSE)),
    "asset_universe": list(EVAL_ASSET_UNIVERSE),
    "active_feature_columns_count": int(len(trained_active_feature_columns)),
    "disabled_feature_count": int(len(eval_disabled)),
    "state_dim": int(agent_params.get("state_dim", 0) or 0),
    "asset_feature_dim": int(state_layout.get("asset_feature_dim", agent_params.get("asset_feature_dim", 0)) or 0),
    "global_feature_dim": int(state_layout.get("global_feature_dim", agent_params.get("global_feature_dim", 0)) or 0),
    "recurrent_memory_enabled": bool(agent_params.get("recurrent_memory_enabled", False)),
    "regime_conditioning_enabled": bool(agent_params.get("regime_conditioning_enabled", False)),
    "state_augmentation_enabled": bool(agent_params.get("state_augmentation_enabled", False)),
    "distributional_critic_enabled": bool(agent_params.get("distributional_critic_enabled", False)),
    "distributional_num_quantiles": int(agent_params.get("distributional_num_quantiles", 0) or 0),
    "eval_action_execution_beta": training_params.get("evaluation_action_execution_beta"),
    "eval_turnover_penalty_scalar": training_params.get("evaluation_turnover_penalty_scalar"),
    "deterministic_eval_mode": EVAL_DETERMINISTIC_MODE,
    "stochastic_eval_mode": EVAL_STOCHASTIC_MODE,
    "resume_enabled": bool(EVAL_ALLOW_RESUME),
}

eval_config_snapshot_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_eval_config_snapshot.json"
hyperparameter_snapshot_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_hyperparameter_snapshot.json"
eval_healthcheck_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_eval_healthcheck.json"

_safe_write_json(eval_config_snapshot_path, eval_config)
_safe_write_json(hyperparameter_snapshot_path, hyperparameter_snapshot)
_safe_write_json(eval_healthcheck_path, eval_healthcheck)

print("[OK] Saved eval config snapshot:", eval_config_snapshot_path)
print("[OK] Saved hyperparameter snapshot:", hyperparameter_snapshot_path)
print("[OK] Saved eval healthcheck:", eval_healthcheck_path)
print("[HEALTH] test window:", eval_healthcheck["test_date_min"], "->", eval_healthcheck["test_date_max"], f"({eval_healthcheck['test_days']} days)")
print("[HEALTH] features:", eval_healthcheck["active_feature_columns_count"], "active |", eval_healthcheck["disabled_feature_count"], "disabled")
print("[HEALTH] eval beta:", eval_healthcheck["eval_action_execution_beta"], "| distributional critic:", eval_healthcheck["distributional_critic_enabled"], f"({eval_healthcheck['distributional_num_quantiles']} quantiles)")


---
## 8) Locate checkpoint


In [69]:
hw_dir = EVAL_RESULTS_ROOT / "high_watermark_checkpoints"
actor_path = hw_dir / f"{CHECKPOINT_TAG}_actor.weights.h5"
critic_path = hw_dir / f"{CHECKPOINT_TAG}_critic.weights.h5"
assert actor_path.exists(), f"Missing: {actor_path}"
assert critic_path.exists(), f"Missing: {critic_path}"
experiment6 = create_experiment6_result_stub(
    exp_idx=6,
    results_root=str(EVAL_RESULTS_ROOT),
    random_seed=EVAL_RANDOM_SEED,
    checkpoint_path=str(hw_dir / CHECKPOINT_TAG),
    base_agent_params=eval_config.get("agent_params", {}),
)
print(f"[OK] Checkpoint: {CHECKPOINT_TAG}")

[create_experiment6_result_stub] Ignoring unsupported kwargs: results_root
[OK] Checkpoint: exp6_tape_hw_ep00404_shp1p162


## 9) Deterministic Horizons + Benchmarks


In [ ]:
import io
import math
import contextlib
import subprocess
import sys


HORIZON_ORDER = list(EVAL_DET_HORIZONS.keys())
COLORS = {
    "model": "#1f77b4",
    "SPY": "#ff7f0e",
    "EqualWeight": "#2ca02c",
    "stratified": "#4C72B0",
    "monte_carlo": "#C44E52",
}
STATE_DIR = OUTPUT_DIR / "_resume_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)


def _unique_test_dates(base_phase1):
    t = base_phase1.test_df.copy()
    t["Date"] = pd.to_datetime(t["Date"])
    return pd.Series(t["Date"].dropna().unique()).sort_values().reset_index(drop=True)


def _find_start_idx(base_phase1, start_date):
    start_date = pd.to_datetime(start_date)
    u = _unique_test_dates(base_phase1)
    matches = np.where(u.values == np.datetime64(start_date))[0]
    if len(matches) == 0:
        future = np.where(u.values >= np.datetime64(start_date))[0]
        if len(future) == 0:
            raise ValueError(f"No valid test date on/after {start_date.date()}")
        return int(future[0])
    return int(matches[0])


def _make_phase1_slice_from_start_idx(base_phase1, start_idx: int, horizon_days: int, *, require_full_window=True):
    t = base_phase1.test_df.copy()
    t["Date"] = pd.to_datetime(t["Date"])
    u = _unique_test_dates(base_phase1)

    if start_idx < 0 or start_idx >= len(u):
        return None, None

    end_idx = min(len(u), start_idx + int(horizon_days))
    win = u.iloc[start_idx:end_idx]
    if require_full_window and len(win) < int(horizon_days):
        return None, None
    if len(win) == 0:
        return None, None

    d0 = pd.to_datetime(win.iloc[0])
    d1 = pd.to_datetime(win.iloc[-1])
    sliced = t[(t["Date"] >= d0) & (t["Date"] <= d1)].copy()
    if sliced.empty:
        return None, None

    phase1_slice = copy.deepcopy(base_phase1)
    phase1_slice.test_df = sliced
    phase1_slice.test_start_date = d0
    meta = {
        "start_idx": int(start_idx),
        "start_date": str(d0.date()),
        "end_date": str(d1.date()),
        "n_days": int(len(win)),
    }
    return phase1_slice, meta


def _run_eval_quiet(**kwargs):
    buf_out, buf_err = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(buf_out), contextlib.redirect_stderr(buf_err):
        er = evaluate_experiment6_checkpoint(**kwargs)
    return er


def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _num_series(df, candidates, mult=1.0):
    c = _pick_col(df, candidates)
    if c is None:
        return pd.Series(dtype=float)
    return pd.to_numeric(df[c], errors="coerce") * mult


def _fmt(x, d=2, suffix=""):
    try:
        xv = float(x)
    except Exception:
        return "n/a"
    if pd.isna(xv):
        return "n/a"
    return f"{xv:.{d}f}{suffix}"


def _pctile(s, q):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return np.nan if len(s) == 0 else float(np.percentile(s, q))


def _cvar_left_tail(s, alpha=0.10):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    cutoff = np.percentile(s, alpha * 100.0)
    tail = s[s <= cutoff]
    return float(tail.mean()) if len(tail) else np.nan


def _compute_return_metrics(return_series: pd.Series):
    s = pd.to_numeric(return_series, errors="coerce").dropna()
    if s.empty:
        return {
            "days_traded": 0,
            "total_return_pct": np.nan,
            "ann_return_pct": np.nan,
            "sharpe_ratio": np.nan,
            "sortino_ratio": np.nan,
            "max_dd_pct": np.nan,
            "volatility_ann": np.nan,
            "win_rate": np.nan,
        }

    wealth = (1.0 + s).cumprod()
    total_return = float(wealth.iloc[-1] - 1.0)
    ann_return = float((wealth.iloc[-1] ** (252.0 / len(s))) - 1.0) if len(s) > 0 else np.nan
    vol = float(s.std(ddof=1) * math.sqrt(252.0)) if len(s) > 1 else np.nan
    sharpe = float((s.mean() / s.std(ddof=1)) * math.sqrt(252.0)) if len(s) > 1 and float(s.std(ddof=1)) > 0 else np.nan
    downside = s[s < 0]
    sortino = float((s.mean() / downside.std(ddof=1)) * math.sqrt(252.0)) if len(downside) > 1 and float(downside.std(ddof=1)) > 0 else np.nan
    drawdown = wealth / wealth.cummax() - 1.0
    max_dd = float(drawdown.min())
    win_rate = float((s > 0).mean())

    return {
        "days_traded": int(len(s)),
        "total_return_pct": total_return * 100.0,
        "ann_return_pct": ann_return * 100.0 if pd.notna(ann_return) else np.nan,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_dd_pct": abs(max_dd) * 100.0 if pd.notna(max_dd) else np.nan,
        "volatility_ann": vol * 100.0 if pd.notna(vol) else np.nan,
        "win_rate": win_rate,
    }


def _ensure_yfinance():
    try:
        import yfinance as yf  # type: ignore
        return yf
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
        import yfinance as yf  # type: ignore
        return yf


def _load_spy_close(start_date, end_date):
    yf = _ensure_yfinance()
    spy = yf.download(
        "SPY",
        start=str(pd.to_datetime(start_date).date()),
        end=str((pd.to_datetime(end_date) + pd.Timedelta(days=5)).date()),
        auto_adjust=True,
        progress=False,
    )
    if spy.empty:
        return pd.Series(dtype=float)
    if isinstance(spy.columns, pd.MultiIndex):
        close = spy["Close"].iloc[:, 0]
    else:
        close = spy["Close"]
    close.index = pd.to_datetime(close.index).tz_localize(None)
    return pd.to_numeric(close, errors="coerce").dropna()


def _build_close_panel(window_df, tickers):
    req = {"Date", "Ticker", "Close"}
    if not req.issubset(window_df.columns):
        missing = sorted(req - set(window_df.columns))
        raise KeyError(f"Window data missing required columns: {missing}")

    local = window_df[["Date", "Ticker", "Close"]].copy()
    local["Date"] = pd.to_datetime(local["Date"])
    local = local[local["Ticker"].isin(tickers)].copy()
    panel = (
        local.pivot_table(index="Date", columns="Ticker", values="Close", aggfunc="last")
        .sort_index()
        .ffill()
    )
    panel = panel.reindex(columns=tickers)
    return panel


def _build_equal_weight_returns(window_df, tickers):
    panel = _build_close_panel(window_df, tickers)
    rets = panel.pct_change().replace([np.inf, -np.inf], np.nan)
    ew = rets.mean(axis=1, skipna=True).dropna()
    return ew


def _build_asset_benchmark_rows(window_df, tickers, horizon_label, spy_total_return_pct):
    panel = _build_close_panel(window_df, tickers)
    out = []
    for ticker in tickers:
        px = pd.to_numeric(panel[ticker], errors="coerce").dropna() if ticker in panel.columns else pd.Series(dtype=float)
        if len(px) < 2:
            continue
        total_ret = float(px.iloc[-1] / px.iloc[0] - 1.0) * 100.0
        ann_ret = float((px.iloc[-1] / px.iloc[0]) ** (252.0 / (len(px) - 1)) - 1.0) * 100.0
        out.append({
            "horizon": horizon_label,
            "ticker": ticker,
            "asset_total_return_pct": total_ret,
            "asset_ann_return_pct": ann_ret,
            "asset_excess_vs_spy_pct": total_ret - float(spy_total_return_pct),
        })
    return out


def _extract_det_row(er, horizon_label, meta):
    dm = er.deterministic_metrics or {}
    turnover_val = dm.get("turnover", np.nan)
    if pd.notna(turnover_val):
        turnover_val = float(turnover_val) * 100.0 if float(turnover_val) <= 2.0 else float(turnover_val)

    return {
        "strategy": "model",
        "horizon": horizon_label,
        "horizon_days": int(meta["n_days"]),
        "start_date": meta["start_date"],
        "end_date": meta["end_date"],
        "days_traded": int(dm.get("episode_length", dm.get("days_traded", meta["n_days"]))),
        "total_return_pct": float(dm.get("total_return_pct", dm.get("total_return", np.nan) * 100.0 if dm.get("total_return") is not None else np.nan)),
        "ann_return_pct": float(dm.get("annualized_return_pct", dm.get("annualized_return", np.nan) * 100.0 if dm.get("annualized_return") is not None else np.nan)),
        "sharpe_ratio": float(dm.get("sharpe_ratio", np.nan)),
        "sortino_ratio": float(dm.get("sortino_ratio", np.nan)),
        "max_dd_pct": float(
            dm.get(
                "max_drawdown_pct",
                dm.get("max_drawdown_abs", dm.get("max_drawdown", np.nan)) * 100.0
                if dm.get("max_drawdown_abs", dm.get("max_drawdown")) is not None
                else np.nan,
            )
        ),
        "volatility_ann": float(
            dm.get(
                "volatility_ann",
                dm.get("volatility_annualized", dm.get("volatility", np.nan)) * 100.0
                if dm.get("volatility_annualized", dm.get("volatility")) is not None
                else np.nan,
            )
        ),
        "turnover_pct": turnover_val,
        "win_rate": float(dm.get("win_rate", np.nan)),
        "market_regime": dm.get("market_regime"),
        "action_uniques": int(dm.get("action_uniques", 0)) if dm.get("action_uniques") is not None else 0,
        "alpha_le1_fraction": float(dm.get("alpha_le1_fraction", np.nan)),
        "alpha_le_1_frac": float(dm.get("alpha_le1_fraction", np.nan)),
        "argmax_alpha_uniques": int(dm.get("argmax_alpha_uniques", 0)) if dm.get("argmax_alpha_uniques") is not None else 0,
    }


def _build_det_daily(er, horizon_label, meta):
    pv = np.asarray(er.deterministic_portfolio) if er.deterministic_portfolio is not None else np.empty((0,))
    dw = np.asarray(er.deterministic_weights) if er.deterministic_weights is not None else np.empty((0, 0))
    da = np.asarray(er.deterministic_alphas) if er.deterministic_alphas is not None else np.empty((0, 0))

    test_df_eval = getattr(er.env_test_deterministic, "processed_data", pd.DataFrame()).copy()
    if isinstance(test_df_eval, pd.DataFrame) and "Date" in test_df_eval.columns:
        dates = (
            pd.to_datetime(test_df_eval["Date"])
            .dropna()
            .drop_duplicates()
            .sort_values()
            .reset_index(drop=True)
        )
    else:
        dates = pd.Series(pd.NaT, index=np.arange(len(pv)))

    n = len(pv)
    if dw.ndim == 2 and dw.shape[0] > 0:
        n = min(n, dw.shape[0])
    if da.ndim == 2 and da.shape[0] > 0:
        n = min(n, da.shape[0])
    if len(dates) > 0:
        n = min(n, len(dates))

    out = pd.DataFrame({
        "horizon": horizon_label,
        "step": np.arange(n),
        "date": pd.to_datetime(dates.iloc[:n].values if len(dates) >= n else pd.NaT, errors="coerce"),
        "portfolio_value": pv[:n],
    })
    out["daily_return"] = out["portfolio_value"].pct_change().fillna(0.0)
    out["cumulative_return"] = out["portfolio_value"] / out["portfolio_value"].iloc[0] - 1.0
    running_max = out["portfolio_value"].cummax()
    out["drawdown"] = out["portfolio_value"] / running_max.replace(0, np.nan) - 1.0
    out["window_start_date"] = meta["start_date"]
    out["window_end_date"] = meta["end_date"]

    if dw.ndim == 2 and dw.shape[0] >= n:
        for i, ticker in enumerate(EVAL_ASSET_UNIVERSE):
            if i < dw.shape[1]:
                out[f"w_{ticker}"] = dw[:n, i]
        if dw.shape[1] > len(EVAL_ASSET_UNIVERSE):
            out["w_cash"] = dw[:n, -1]

    if da.ndim == 2 and da.shape[0] >= n:
        for i, ticker in enumerate(EVAL_ASSET_UNIVERSE):
            if i < da.shape[1]:
                out[f"alpha_{ticker}"] = da[:n, i]
        if da.shape[1] > len(EVAL_ASSET_UNIVERSE):
            out["alpha_cash"] = da[:n, -1]

    return out


det_results = []
det_daily_data = {}
det_benchmark_rows = []
det_asset_vs_spy_rows = []
det_resume_path = STATE_DIR / f"{ARTIFACT_TAG}_deterministic_summary.partial.csv"
det_benchmark_resume_path = STATE_DIR / f"{ARTIFACT_TAG}_deterministic_benchmarks.partial.csv"
det_asset_resume_path = STATE_DIR / f"{ARTIFACT_TAG}_deterministic_asset_vs_spy.partial.csv"
det_daily_resume_dir = STATE_DIR / "deterministic_daily"
det_daily_resume_dir.mkdir(parents=True, exist_ok=True)

if EVAL_ALLOW_RESUME and det_resume_path.exists():
    det_results = pd.read_csv(det_resume_path).to_dict(orient="records")
if EVAL_ALLOW_RESUME and det_benchmark_resume_path.exists():
    det_benchmark_rows = pd.read_csv(det_benchmark_resume_path).to_dict(orient="records")
if EVAL_ALLOW_RESUME and det_asset_resume_path.exists():
    det_asset_vs_spy_rows = pd.read_csv(det_asset_resume_path).to_dict(orient="records")

test_start_idx = _find_start_idx(eval_phase1_data, EVAL_FORCE_TEST_START_DATE)
test_dates_all = _unique_test_dates(eval_phase1_data)
spy_close_full = _load_spy_close(test_dates_all.iloc[test_start_idx], test_dates_all.iloc[-1])

print("=" * 80)
print("DETERMINISTIC HORIZON SWEEP (Run10 test split)")
print("=" * 80)

for horizon_label, horizon_days in EVAL_DET_HORIZONS.items():
    completed_horizons = {str(r.get("horizon")) for r in det_results}
    if EVAL_ALLOW_RESUME and horizon_label in completed_horizons:
        daily_fp = det_daily_resume_dir / f"{ARTIFACT_TAG}_det_{horizon_label}_daily.csv"
        if daily_fp.exists():
            det_daily_data[horizon_label] = pd.read_csv(daily_fp, parse_dates=["date"])
        print(f"[SKIP] {horizon_label}: deterministic results already present in resume state")
        continue

    phase1_slice, meta = _make_phase1_slice_from_start_idx(
        eval_phase1_data,
        test_start_idx,
        horizon_days,
        require_full_window=True,
    )
    if phase1_slice is None:
        print(f"[WARN] {horizon_label}: no valid deterministic slice")
        continue

    er = _run_eval_quiet(
        experiment6=experiment6,
        phase1_data=phase1_slice,
        config=eval_config,
        random_seed=EVAL_RANDOM_SEED,
        checkpoint_path_override=str(hw_dir / CHECKPOINT_TAG),
        deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
        stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
        num_eval_runs=0,
        stochastic_episode_length_limit=int(meta["n_days"]),
        save_eval_logs=EVAL_SAVE_LOGS,
        save_eval_artifacts=EVAL_SAVE_ARTIFACTS,
    )

    det_row = _extract_det_row(er, horizon_label, meta)
    det_results.append(det_row)
    det_daily = _build_det_daily(er, horizon_label, meta)
    det_daily_data[horizon_label] = det_daily

    model_eval_dates = pd.to_datetime(det_daily["date"]).dropna()
    if model_eval_dates.empty:
        model_eval_dates = pd.Series(pd.to_datetime(pd.date_range(meta["start_date"], periods=int(meta["n_days"]), freq="B")))

    window_df = phase1_slice.test_df.copy()
    ew_returns = _build_equal_weight_returns(window_df, EVAL_ASSET_UNIVERSE)
    ew_returns = ew_returns.reindex(pd.DatetimeIndex(model_eval_dates)).dropna()
    ew_metrics = _compute_return_metrics(ew_returns)

    spy_close = spy_close_full.loc[(spy_close_full.index >= pd.to_datetime(meta["start_date"])) & (spy_close_full.index <= pd.to_datetime(meta["end_date"]))].copy()
    spy_returns = spy_close.pct_change().dropna()
    spy_returns = spy_returns.reindex(pd.DatetimeIndex(model_eval_dates)).dropna()
    spy_metrics = _compute_return_metrics(spy_returns)

    for strategy_name, metrics in [("SPY", spy_metrics), ("EqualWeight", ew_metrics)]:
        det_benchmark_rows.append({
            "strategy": strategy_name,
            "horizon": horizon_label,
            "horizon_days": int(meta["n_days"]),
            "start_date": meta["start_date"],
            "end_date": meta["end_date"],
            **metrics,
            "turnover_pct": np.nan,
            "market_regime": det_row.get("market_regime"),
        })

    det_benchmark_rows.append(det_row.copy())
    det_asset_vs_spy_rows.extend(
        _build_asset_benchmark_rows(window_df, EVAL_ASSET_UNIVERSE, horizon_label, spy_metrics.get("total_return_pct", np.nan))
    )

    pd.DataFrame(det_results).to_csv(det_resume_path, index=False)
    pd.DataFrame(det_benchmark_rows).to_csv(det_benchmark_resume_path, index=False)
    pd.DataFrame(det_asset_vs_spy_rows).to_csv(det_asset_resume_path, index=False)
    det_daily.to_csv(det_daily_resume_dir / f"{ARTIFACT_TAG}_det_{horizon_label}_daily.csv", index=False)

    avg_weight = np.nan
    if not det_daily.empty:
        weight_cols = [c for c in det_daily.columns if c.startswith("w_")]
        if weight_cols:
            avg_weight = float(det_daily[weight_cols].sum(axis=1).mean())

    print(
        f"[DET] {horizon_label} start={meta['start_date']} end={meta['end_date']} | "
        f"model sh={_fmt(det_row['sharpe_ratio'], 4)} ret={_fmt(det_row['total_return_pct'], 2, '%')} mdd={_fmt(det_row['max_dd_pct'], 2, '%')} | "
        f"SPY sh={_fmt(spy_metrics['sharpe_ratio'], 4)} ret={_fmt(spy_metrics['total_return_pct'], 2, '%')} | "
        f"EW sh={_fmt(ew_metrics['sharpe_ratio'], 4)} ret={_fmt(ew_metrics['total_return_pct'], 2, '%')}"
    )

det_results_df = pd.DataFrame(det_results).sort_values("horizon")
det_benchmark_long_df = pd.DataFrame(det_benchmark_rows)
det_asset_vs_spy_df = pd.DataFrame(det_asset_vs_spy_rows)

det_comparison_rows = []
for horizon_label in HORIZON_ORDER:
    model_row = det_benchmark_long_df[(det_benchmark_long_df["horizon"] == horizon_label) & (det_benchmark_long_df["strategy"] == "model")]
    spy_row = det_benchmark_long_df[(det_benchmark_long_df["horizon"] == horizon_label) & (det_benchmark_long_df["strategy"] == "SPY")]
    ew_row = det_benchmark_long_df[(det_benchmark_long_df["horizon"] == horizon_label) & (det_benchmark_long_df["strategy"] == "EqualWeight")]
    if model_row.empty:
        continue
    model_row = model_row.iloc[0]
    spy_row = spy_row.iloc[0] if not spy_row.empty else pd.Series(dtype=object)
    ew_row = ew_row.iloc[0] if not ew_row.empty else pd.Series(dtype=object)
    det_comparison_rows.append({
        "horizon": horizon_label,
        "start_date": model_row.get("start_date"),
        "end_date": model_row.get("end_date"),
        "model_sharpe": model_row.get("sharpe_ratio"),
        "model_total_return_pct": model_row.get("total_return_pct"),
        "model_ann_return_pct": model_row.get("ann_return_pct"),
        "model_mdd_pct": model_row.get("max_dd_pct"),
        "spy_sharpe": spy_row.get("sharpe_ratio"),
        "spy_total_return_pct": spy_row.get("total_return_pct"),
        "spy_mdd_pct": spy_row.get("max_dd_pct"),
        "ew_sharpe": ew_row.get("sharpe_ratio"),
        "ew_total_return_pct": ew_row.get("total_return_pct"),
        "ew_mdd_pct": ew_row.get("max_dd_pct"),
        "excess_vs_spy_pct": model_row.get("total_return_pct") - spy_row.get("total_return_pct", np.nan),
        "excess_vs_ew_pct": model_row.get("total_return_pct") - ew_row.get("total_return_pct", np.nan),
    })

det_comparison_df = pd.DataFrame(det_comparison_rows).sort_values("horizon")

display(det_results_df)
display(det_comparison_df)


In [ ]:
det_csv_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_summary.csv"
det_results_df.to_csv(det_csv_path, index=False)

det_benchmark_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_benchmarks_long.csv"
det_benchmark_long_df.to_csv(det_benchmark_path, index=False)

det_comparison_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_comparison.csv"
det_comparison_df.to_csv(det_comparison_path, index=False)

det_asset_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_asset_vs_spy.csv"
det_asset_vs_spy_df.to_csv(det_asset_path, index=False)

det_daily_dir = OUTPUT_DIR / "deterministic_daily"
det_daily_dir.mkdir(parents=True, exist_ok=True)
for horizon_label, df in det_daily_data.items():
    df.to_csv(det_daily_dir / f"{ARTIFACT_TAG}_det_{horizon_label}_daily.csv", index=False)

print(f"[OK] Saved deterministic outputs to {OUTPUT_DIR}")


## 10) Stratified Stochastic Robustness


In [ ]:
def _sample_stratified_start_indices(n_valid: int, n_samples: int, n_strata: int, seed: int):
    rng = np.random.default_rng(seed)
    n_samples = min(int(n_samples), int(n_valid))
    n_strata = max(1, min(int(n_strata), int(n_valid)))
    edges = np.linspace(0, n_valid, n_strata + 1, dtype=int)
    base = n_samples // n_strata
    rem = n_samples % n_strata
    picks = []
    for s in range(n_strata):
        lo, hi = edges[s], edges[s + 1]
        pool = np.arange(lo, hi)
        if len(pool) == 0:
            continue
        k = min(len(pool), base + (1 if s < rem else 0))
        if k > 0:
            picks.extend(rng.choice(pool, size=k, replace=False).tolist())
    if len(picks) < n_samples:
        remaining = np.setdiff1d(np.arange(n_valid), np.array(picks, dtype=int), assume_unique=False)
        need = n_samples - len(picks)
        if len(remaining) > 0:
            picks.extend(rng.choice(remaining, size=min(need, len(remaining)), replace=False).tolist())
    return sorted(picks[:n_samples])


def _extract_stochastic_row(sto: pd.DataFrame):
    row = sto.iloc[0].copy()
    if "total_return_pct" not in row.index and "total_return" in row.index:
        row["total_return_pct"] = pd.to_numeric(row["total_return"], errors="coerce") * 100.0
    if "ann_return_pct" not in row.index:
        if "annualized_return" in row.index:
            row["ann_return_pct"] = pd.to_numeric(row["annualized_return"], errors="coerce") * 100.0
        elif "annualized_return_pct" in row.index:
            row["ann_return_pct"] = pd.to_numeric(row["annualized_return_pct"], errors="coerce")
    if "max_dd_pct" not in row.index:
        if "max_drawdown" in row.index:
            row["max_dd_pct"] = pd.to_numeric(row["max_drawdown"], errors="coerce") * 100.0
        elif "max_drawdown_abs" in row.index:
            row["max_dd_pct"] = pd.to_numeric(row["max_drawdown_abs"], errors="coerce") * 100.0
    if "volatility_ann" not in row.index:
        if "volatility" in row.index:
            row["volatility_ann"] = pd.to_numeric(row["volatility"], errors="coerce") * 100.0
        elif "volatility_annualized" in row.index:
            row["volatility_ann"] = pd.to_numeric(row["volatility_annualized"], errors="coerce") * 100.0
    return row


stratified_stoch_rows = []
stratified_stoch_daily = {}
stratified_by_horizon = {}
u_all = _unique_test_dates(eval_phase1_data)
stratified_resume_dir = STATE_DIR / "stratified"
stratified_resume_dir.mkdir(parents=True, exist_ok=True)
stratified_daily_resume_dir = stratified_resume_dir / "daily"
stratified_daily_resume_dir.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("STRATIFIED STOCHASTIC ROBUSTNESS")
print("=" * 80)

for h_i, (h_label, h_days) in enumerate(EVAL_STOCHASTIC_HORIZONS.items(), start=1):
    horizon_resume_path = stratified_resume_dir / f"{ARTIFACT_TAG}_stratified_{h_label}.csv"
    if EVAL_ALLOW_RESUME and horizon_resume_path.exists():
        horizon_df = pd.read_csv(horizon_resume_path)
        if len(horizon_df) >= EVAL_STRATIFIED_RUNS_PER_HORIZON:
            stratified_by_horizon[h_label] = horizon_df
            stratified_stoch_rows.extend(horizon_df.to_dict(orient="records"))
            print(f"[SKIP] {h_label}: stratified results already present in resume state")
            continue

    max_start_idx = len(u_all) - int(h_days)
    if max_start_idx < 0:
        print(f"[WARN] {h_label}: horizon too long for available test window")
        continue

    valid_count = max_start_idx + 1
    start_idxs = _sample_stratified_start_indices(
        n_valid=valid_count,
        n_samples=EVAL_STRATIFIED_RUNS_PER_HORIZON,
        n_strata=EVAL_STRATIFIED_STRATA_PER_HORIZON,
        seed=EVAL_RANDOM_SEED + h_i * 10_000 + int(h_days),
    )

    print(f"\n{h_label} ({h_days}d): {len(start_idxs)} stratified starts")
    print(f"valid start pool: {valid_count} | first={u_all.iloc[0].date()} | last={u_all.iloc[max_start_idx].date()}")

    horizon_rows = []
    for run_idx, start_idx in enumerate(start_idxs, start=1):
        phase1_slice, meta = _make_phase1_slice_from_start_idx(eval_phase1_data, start_idx, int(h_days), require_full_window=True)
        if phase1_slice is None:
            continue

        seed = int(EVAL_RANDOM_SEED + int(h_days) * 100_000 + run_idx)
        er = _run_eval_quiet(
            experiment6=experiment6,
            phase1_data=phase1_slice,
            config=eval_config,
            random_seed=seed,
            checkpoint_path_override=str(hw_dir / CHECKPOINT_TAG),
            deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
            stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
            num_eval_runs=1,
            stochastic_episode_length_limit=int(h_days),
            save_eval_logs=EVAL_SAVE_LOGS,
            save_eval_artifacts=EVAL_SAVE_ARTIFACTS,
        )

        sto = er.stochastic_results if isinstance(er.stochastic_results, pd.DataFrame) else pd.DataFrame()
        if sto.empty:
            continue

        row = _extract_stochastic_row(sto).to_dict()
        row["method"] = "stratified"
        row["horizon"] = h_label
        row["horizon_days"] = int(h_days)
        row["sample_id"] = int(run_idx)
        row["sampled_start_idx"] = int(start_idx)
        row["forced_start_date"] = meta["start_date"]
        row["window_end_date"] = meta["end_date"]
        stratified_stoch_rows.append(row)
        horizon_rows.append(row)

        print(
            f"[STRAT {h_label} {run_idx:03d}/{len(start_idxs)}] "
            f"start={row.get('start_date', meta['start_date'])} seed={int(row.get('seed', seed))} | "
            f"tot={_fmt(row.get('total_return_pct', np.nan), 2, '%')} | "
            f"ann={_fmt(row.get('ann_return_pct', np.nan), 2, '%')} | "
            f"sh={_fmt(row.get('sharpe_ratio', row.get('sharpe', np.nan)), 4)} | "
            f"mdd={_fmt(row.get('max_dd_pct', np.nan), 2, '%')} | "
            f"turn={_fmt(row.get('turnover', np.nan), 4)}"
        )

        if er.stochastic_weights and len(er.stochastic_weights) >= 1:
            w = np.asarray(er.stochastic_weights[0])
            a = np.asarray(er.stochastic_actions[0]) if er.stochastic_actions and len(er.stochastic_actions) >= 1 else np.empty((0, 0))
            al = np.asarray(er.stochastic_alphas[0]) if er.stochastic_alphas and len(er.stochastic_alphas) >= 1 else np.empty((0, 0))
            n = len(w)
            dd = pd.DataFrame({"step": np.arange(n), "horizon": h_label, "sample_id": run_idx, "start_date": meta["start_date"]})
            if w.ndim == 2 and w.shape[1] > 0:
                for j, ticker in enumerate(EVAL_ASSET_UNIVERSE):
                    if j < w.shape[1]:
                        dd[f"w_{ticker}"] = w[:, j]
                if w.shape[1] > len(EVAL_ASSET_UNIVERSE):
                    dd["w_cash"] = w[:, -1]
            if a.ndim == 2 and len(a) == n:
                for j in range(a.shape[1]):
                    dd[f"a_{j}"] = a[:, j]
            if al.ndim == 2 and len(al) == n:
                for j in range(al.shape[1]):
                    dd[f"alpha_{j}"] = al[:, j]
            stratified_stoch_daily[(h_label, run_idx)] = dd
            dd.to_csv(stratified_daily_resume_dir / f"{ARTIFACT_TAG}_stratified_{h_label}_{run_idx:03d}.csv", index=False)

    horizon_df = pd.DataFrame(horizon_rows)
    stratified_by_horizon[h_label] = horizon_df
    horizon_df.to_csv(horizon_resume_path, index=False)
    if not horizon_df.empty:
        print(f"[STRAT {h_label}] Sharpe mean±std: {horizon_df['sharpe_ratio'].mean():.4f} ± {horizon_df['sharpe_ratio'].std():.4f}")

stratified_stoch_df = pd.DataFrame(stratified_stoch_rows)
display(stratified_stoch_df.head())


In [ ]:
stratified_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_stratified_stochastic_all.csv"
stratified_stoch_df.to_csv(stratified_path, index=False)

stratified_daily_dir = OUTPUT_DIR / "stratified_stochastic_daily"
stratified_daily_dir.mkdir(parents=True, exist_ok=True)
for (horizon_label, sample_id), df in stratified_stoch_daily.items():
    df.to_csv(stratified_daily_dir / f"{ARTIFACT_TAG}_stratified_{horizon_label}_{sample_id:03d}_daily.csv", index=False)

print(f"[OK] Saved stratified stochastic outputs to {OUTPUT_DIR}")


In [ ]:
def _sample_unconstrained_start_indices(n_valid: int, n_samples: int, seed: int):
    rng = np.random.default_rng(seed)
    n_samples = min(int(n_samples), int(n_valid))
    if n_samples <= 0:
        return []
    return sorted(rng.choice(np.arange(n_valid), size=n_samples, replace=False).tolist())


mc_stoch_rows = []
mc_stoch_daily = {}
mc_by_horizon = {}
mc_resume_dir = STATE_DIR / "monte_carlo"
mc_resume_dir.mkdir(parents=True, exist_ok=True)
mc_daily_resume_dir = mc_resume_dir / "daily"
mc_daily_resume_dir.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("UNCONSTRAINED MONTE CARLO START-DATE DRAWS")
print("=" * 80)

for h_i, (h_label, h_days) in enumerate(EVAL_STOCHASTIC_HORIZONS.items(), start=1):
    horizon_resume_path = mc_resume_dir / f"{ARTIFACT_TAG}_mc_{h_label}.csv"
    if EVAL_ALLOW_RESUME and horizon_resume_path.exists():
        horizon_df = pd.read_csv(horizon_resume_path)
        if len(horizon_df) >= EVAL_MONTE_CARLO_RUNS_PER_HORIZON:
            mc_by_horizon[h_label] = horizon_df
            mc_stoch_rows.extend(horizon_df.to_dict(orient="records"))
            print(f"[SKIP] {h_label}: Monte Carlo results already present in resume state")
            continue

    max_start_idx = len(u_all) - int(h_days)
    if max_start_idx < 0:
        print(f"[WARN] {h_label}: horizon too long for available test window")
        continue

    valid_count = max_start_idx + 1
    start_idxs = _sample_unconstrained_start_indices(
        n_valid=valid_count,
        n_samples=EVAL_MONTE_CARLO_RUNS_PER_HORIZON,
        seed=EVAL_RANDOM_SEED + h_i * 20_000 + int(h_days),
    )

    print(f"\n{h_label} ({h_days}d): {len(start_idxs)} unconstrained draws")
    print(f"valid start pool: {valid_count} | first={u_all.iloc[0].date()} | last={u_all.iloc[max_start_idx].date()}")

    horizon_rows = []
    for run_idx, start_idx in enumerate(start_idxs, start=1):
        phase1_slice, meta = _make_phase1_slice_from_start_idx(eval_phase1_data, start_idx, int(h_days), require_full_window=True)
        if phase1_slice is None:
            continue

        seed = int(EVAL_RANDOM_SEED + 9_000_000 + int(h_days) * 100_000 + run_idx)
        er = _run_eval_quiet(
            experiment6=experiment6,
            phase1_data=phase1_slice,
            config=eval_config,
            random_seed=seed,
            checkpoint_path_override=str(hw_dir / CHECKPOINT_TAG),
            deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
            stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
            num_eval_runs=1,
            stochastic_episode_length_limit=int(h_days),
            save_eval_logs=EVAL_SAVE_LOGS,
            save_eval_artifacts=EVAL_SAVE_ARTIFACTS,
        )

        sto = er.stochastic_results if isinstance(er.stochastic_results, pd.DataFrame) else pd.DataFrame()
        if sto.empty:
            continue

        row = _extract_stochastic_row(sto).to_dict()
        row["method"] = "monte_carlo"
        row["horizon"] = h_label
        row["horizon_days"] = int(h_days)
        row["sample_id"] = int(run_idx)
        row["sampled_start_idx"] = int(start_idx)
        row["forced_start_date"] = meta["start_date"]
        row["window_end_date"] = meta["end_date"]
        mc_stoch_rows.append(row)
        horizon_rows.append(row)

        print(
            f"[MC {h_label} {run_idx:03d}/{len(start_idxs)}] "
            f"start={row.get('start_date', meta['start_date'])} seed={int(row.get('seed', seed))} | "
            f"tot={_fmt(row.get('total_return_pct', np.nan), 2, '%')} | "
            f"ann={_fmt(row.get('ann_return_pct', np.nan), 2, '%')} | "
            f"sh={_fmt(row.get('sharpe_ratio', row.get('sharpe', np.nan)), 4)} | "
            f"mdd={_fmt(row.get('max_dd_pct', np.nan), 2, '%')} | "
            f"turn={_fmt(row.get('turnover', np.nan), 4)}"
        )

        if er.stochastic_weights and len(er.stochastic_weights) >= 1:
            w = np.asarray(er.stochastic_weights[0])
            a = np.asarray(er.stochastic_actions[0]) if er.stochastic_actions and len(er.stochastic_actions) >= 1 else np.empty((0, 0))
            al = np.asarray(er.stochastic_alphas[0]) if er.stochastic_alphas and len(er.stochastic_alphas) >= 1 else np.empty((0, 0))
            n = len(w)
            dd = pd.DataFrame({"step": np.arange(n), "horizon": h_label, "sample_id": run_idx, "start_date": meta["start_date"]})
            if w.ndim == 2 and w.shape[1] > 0:
                for j, ticker in enumerate(EVAL_ASSET_UNIVERSE):
                    if j < w.shape[1]:
                        dd[f"w_{ticker}"] = w[:, j]
                if w.shape[1] > len(EVAL_ASSET_UNIVERSE):
                    dd["w_cash"] = w[:, -1]
            if a.ndim == 2 and len(a) == n:
                for j in range(a.shape[1]):
                    dd[f"a_{j}"] = a[:, j]
            if al.ndim == 2 and len(al) == n:
                for j in range(al.shape[1]):
                    dd[f"alpha_{j}"] = al[:, j]
            mc_stoch_daily[(h_label, run_idx)] = dd
            dd.to_csv(mc_daily_resume_dir / f"{ARTIFACT_TAG}_mc_{h_label}_{run_idx:03d}.csv", index=False)

    horizon_df = pd.DataFrame(horizon_rows)
    mc_by_horizon[h_label] = horizon_df
    horizon_df.to_csv(horizon_resume_path, index=False)
    if not horizon_df.empty:
        print(f"[MC {h_label}] Sharpe mean±std: {horizon_df['sharpe_ratio'].mean():.4f} ± {horizon_df['sharpe_ratio'].std():.4f}")

mc_stoch_df = pd.DataFrame(mc_stoch_rows)
mc_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_monte_carlo_stochastic_all.csv"
mc_stoch_df.to_csv(mc_path, index=False)

mc_daily_dir = OUTPUT_DIR / "monte_carlo_stochastic_daily"
mc_daily_dir.mkdir(parents=True, exist_ok=True)
for (horizon_label, sample_id), df in mc_stoch_daily.items():
    df.to_csv(mc_daily_dir / f"{ARTIFACT_TAG}_mc_{horizon_label}_{sample_id:03d}_daily.csv", index=False)

display(mc_stoch_df.head())
print(f"[OK] Saved Monte Carlo stochastic outputs to {OUTPUT_DIR}")


## 11) Comparative Robustness Analysis


In [ ]:
def _summarize_stochastic_method(df: pd.DataFrame, method_name: str):
    rows = []
    if df.empty:
        return pd.DataFrame(rows)
    for horizon_label, g in df.groupby("horizon"):
        sharpe = _num_series(g, ["sharpe_ratio", "sharpe"])
        sortino = _num_series(g, ["sortino_ratio", "sortino"])
        ann_ret = _num_series(g, ["ann_return_pct", "annualized_return", "annualized_return_pct"], mult=(1.0 if "ann_return_pct" in g.columns else 100.0))
        total_ret = _num_series(g, ["total_return_pct", "total_return"], mult=(1.0 if "total_return_pct" in g.columns else 100.0))
        mdd = _num_series(g, ["max_dd_pct", "max_drawdown", "max_drawdown_abs"], mult=(1.0 if "max_dd_pct" in g.columns else 100.0))
        turnover = _num_series(g, ["turnover"])

        rows.append({
            "method": method_name,
            "horizon": horizon_label,
            "n_runs": int(len(g)),
            "ann_ret_mean_pct": ann_ret.mean(),
            "ann_ret_std_pct": ann_ret.std(),
            "ann_ret_p10_pct": _pctile(ann_ret, 10),
            "ann_ret_cvar10_pct": _cvar_left_tail(ann_ret, 0.10),
            "total_ret_mean_pct": total_ret.mean(),
            "total_ret_std_pct": total_ret.std(),
            "sharpe_mean": sharpe.mean(),
            "sharpe_std": sharpe.std(),
            "sharpe_p10": _pctile(sharpe, 10),
            "sortino_mean": sortino.mean(),
            "mdd_mean_pct": mdd.mean(),
            "mdd_std_pct": mdd.std(),
            "mdd_p90_pct": _pctile(mdd, 90),
            "turnover_mean": turnover.mean(),
            "turnover_std": turnover.std(),
            "positive_sharpe_rate_pct": float((sharpe > 0).mean() * 100.0) if len(sharpe.dropna()) else np.nan,
            "sharpe_gt_05_rate_pct": float((sharpe > 0.5).mean() * 100.0) if len(sharpe.dropna()) else np.nan,
            "sharpe_gt_10_rate_pct": float((sharpe > 1.0).mean() * 100.0) if len(sharpe.dropna()) else np.nan,
            "start_date_min": pd.to_datetime(g["forced_start_date"]).min().date(),
            "start_date_max": pd.to_datetime(g["forced_start_date"]).max().date(),
        })
    return pd.DataFrame(rows).sort_values(["horizon", "method"])


stratified_summary_df = _summarize_stochastic_method(stratified_stoch_df, "stratified")
mc_summary_df = _summarize_stochastic_method(mc_stoch_df, "monte_carlo")
stoch_summary_combined_df = pd.concat([stratified_summary_df, mc_summary_df], ignore_index=True)

stoch_runs_combined_df = pd.concat(
    [stratified_stoch_df.copy(), mc_stoch_df.copy()],
    ignore_index=True,
)

stoch_method_compare_df = (
    stoch_summary_combined_df.pivot_table(
        index="horizon",
        columns="method",
        values=["ann_ret_mean_pct", "sharpe_mean", "mdd_mean_pct", "turnover_mean"],
        aggfunc="first",
    )
    if not stoch_summary_combined_df.empty
    else pd.DataFrame()
)

stoch_summary_combined_df.to_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_stochastic_method_summary.csv", index=False)
stoch_runs_combined_df.to_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_stochastic_runs_combined.csv", index=False)

print("=" * 80)
print("STOCHASTIC METHOD SUMMARY")
print("=" * 80)
display(stoch_summary_combined_df)
if not stoch_method_compare_df.empty:
    display(stoch_method_compare_df)


## 12) Generalizability Check


In [ ]:
def _status_high_good(value, good, warn):
    if pd.isna(value):
        return "missing"
    if value >= good:
        return "pass"
    if value >= warn:
        return "watch"
    return "fail"


def _status_low_good(value, good, warn):
    if pd.isna(value):
        return "missing"
    if value <= good:
        return "pass"
    if value <= warn:
        return "watch"
    return "fail"


def _overall_generalizability_verdict(checks_df):
    fail_count = int((checks_df["status"] == "fail").sum())
    watch_count = int((checks_df["status"] == "watch").sum())
    if fail_count >= 3:
        return "fragile"
    if fail_count >= 1 or watch_count >= 4:
        return "mixed"
    return "strong"


def _horizon_sort_key(label):
    order = {k: i for i, k in enumerate(HORIZON_ORDER)}
    return order.get(label, 999)


model_det_df = det_results_df.copy().sort_values("horizon", key=lambda s: s.map(_horizon_sort_key))
bench_pivot = det_benchmark_long_df.pivot_table(index="horizon", columns="strategy", values=["ann_return_pct", "sharpe_ratio", "max_dd_pct"], aggfunc="first")

per_horizon_rows = []
for _, drow in model_det_df.iterrows():
    h = drow["horizon"]
    for method_name in ["stratified", "monte_carlo"]:
        ss = stoch_summary_combined_df[(stoch_summary_combined_df["horizon"] == h) & (stoch_summary_combined_df["method"] == method_name)]
        if ss.empty:
            continue
        srow = ss.iloc[0]
        try:
            spy_ann = bench_pivot.loc[h, ("ann_return_pct", "SPY")]
            ew_ann = bench_pivot.loc[h, ("ann_return_pct", "EqualWeight")]
        except Exception:
            spy_ann = np.nan
            ew_ann = np.nan
        per_horizon_rows.append({
            "horizon": h,
            "method": method_name,
            "det_ann_return_pct": pd.to_numeric(drow.get("ann_return_pct"), errors="coerce"),
            "det_sharpe_ratio": pd.to_numeric(drow.get("sharpe_ratio"), errors="coerce"),
            "det_max_dd_pct": pd.to_numeric(drow.get("max_dd_pct"), errors="coerce"),
            "sto_ann_ret_mean_pct": pd.to_numeric(srow.get("ann_ret_mean_pct"), errors="coerce"),
            "sto_ann_ret_p10_pct": pd.to_numeric(srow.get("ann_ret_p10_pct"), errors="coerce"),
            "sto_ann_ret_cvar10_pct": pd.to_numeric(srow.get("ann_ret_cvar10_pct"), errors="coerce"),
            "sto_sharpe_mean": pd.to_numeric(srow.get("sharpe_mean"), errors="coerce"),
            "sto_sharpe_std": pd.to_numeric(srow.get("sharpe_std"), errors="coerce"),
            "sto_positive_sharpe_rate_pct": pd.to_numeric(srow.get("positive_sharpe_rate_pct"), errors="coerce"),
            "sto_sharpe_gt_05_rate_pct": pd.to_numeric(srow.get("sharpe_gt_05_rate_pct"), errors="coerce"),
            "sto_sharpe_gt_10_rate_pct": pd.to_numeric(srow.get("sharpe_gt_10_rate_pct"), errors="coerce"),
            "sto_mdd_mean_pct": pd.to_numeric(srow.get("mdd_mean_pct"), errors="coerce"),
            "sto_turnover_mean": pd.to_numeric(srow.get("turnover_mean"), errors="coerce"),
            "det_vs_sto_sharpe_gap": pd.to_numeric(drow.get("sharpe_ratio"), errors="coerce") - pd.to_numeric(srow.get("sharpe_mean"), errors="coerce"),
            "det_vs_sto_ann_gap_pct": pd.to_numeric(drow.get("ann_return_pct"), errors="coerce") - pd.to_numeric(srow.get("ann_ret_mean_pct"), errors="coerce"),
            "det_beats_spy_ann": float(pd.to_numeric(drow.get("ann_return_pct"), errors="coerce") > pd.to_numeric(spy_ann, errors="coerce")) if pd.notna(spy_ann) else np.nan,
            "det_beats_equal_weight_ann": float(pd.to_numeric(drow.get("ann_return_pct"), errors="coerce") > pd.to_numeric(ew_ann, errors="coerce")) if pd.notna(ew_ann) else np.nan,
            "action_uniques": pd.to_numeric(drow.get("action_uniques"), errors="coerce"),
            "action_unique_ratio": pd.to_numeric(drow.get("action_uniques"), errors="coerce") / max(pd.to_numeric(drow.get("days_traded"), errors="coerce"), 1.0),
            "alpha_le1_fraction": pd.to_numeric(drow.get("alpha_le1_fraction", drow.get("alpha_le_1_frac")), errors="coerce"),
            "argmax_alpha_uniques": pd.to_numeric(drow.get("argmax_alpha_uniques"), errors="coerce"),
        })

generalizability_by_horizon_df = pd.DataFrame(per_horizon_rows).sort_values(["horizon", "method"], key=lambda s: s.map(_horizon_sort_key) if s.name == "horizon" else s)
generalizability_by_horizon_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_generalizability_by_horizon.csv"
generalizability_by_horizon_df.to_csv(generalizability_by_horizon_path, index=False)

spy_ann = bench_pivot["ann_return_pct"]["SPY"] if ("ann_return_pct" in bench_pivot.columns.get_level_values(0) and "SPY" in bench_pivot["ann_return_pct"].columns) else pd.Series(dtype=float)
ew_ann = bench_pivot["ann_return_pct"]["EqualWeight"] if ("ann_return_pct" in bench_pivot.columns.get_level_values(0) and "EqualWeight" in bench_pivot["ann_return_pct"].columns) else pd.Series(dtype=float)

checks = []

def _add_check(name, value, status, rationale):
    checks.append({"check": name, "value": None if pd.isna(value) else float(value), "status": status, "rationale": rationale})

model_det_sharpe = pd.to_numeric(model_det_df["sharpe_ratio"], errors="coerce")
model_det_ann = pd.to_numeric(model_det_df["ann_return_pct"], errors="coerce")
model_det_mdd = pd.to_numeric(model_det_df["max_dd_pct"], errors="coerce")
model_det_action_ratio = pd.to_numeric(model_det_df.get("action_uniques", pd.Series(dtype=float)), errors="coerce") / pd.to_numeric(model_det_df.get("days_traded", pd.Series(dtype=float)), errors="coerce").replace(0, np.nan)
model_det_alpha_le1 = pd.to_numeric(model_det_df.get("alpha_le1_fraction", model_det_df.get("alpha_le_1_frac", pd.Series(dtype=float))), errors="coerce")
model_det_argmax = pd.to_numeric(model_det_df.get("argmax_alpha_uniques", pd.Series(dtype=float)), errors="coerce")

spy_win_rate = float((model_det_ann.values > pd.to_numeric(spy_ann.reindex(model_det_df["horizon"]).values, errors="coerce")).mean() * 100.0) if len(model_det_df) and len(spy_ann) else np.nan
ew_win_rate = float((model_det_ann.values > pd.to_numeric(ew_ann.reindex(model_det_df["horizon"]).values, errors="coerce")).mean() * 100.0) if len(model_det_df) and len(ew_ann) else np.nan

_add_check("deterministic_vs_spy_win_rate_pct", spy_win_rate, _status_high_good(spy_win_rate, 75.0, 50.0), "Model should beat SPY on most deterministic horizons.")
_add_check("deterministic_vs_equal_weight_win_rate_pct", ew_win_rate, _status_high_good(ew_win_rate, 75.0, 50.0), "Model should beat equal-weight on most deterministic horizons.")
_add_check("deterministic_min_sharpe", model_det_sharpe.min(), _status_high_good(model_det_sharpe.min(), 0.75, 0.35), "Weak worst-horizon Sharpe suggests regime fragility.")
_add_check("deterministic_sharpe_std", model_det_sharpe.std(), _status_low_good(model_det_sharpe.std(), 0.25, 0.45), "Lower Sharpe dispersion across horizons indicates more stable generalization.")
_add_check("deterministic_max_mdd_pct", model_det_mdd.max(), _status_low_good(model_det_mdd.max(), 25.0, 32.0), "Worst deterministic drawdown should stay controlled.")
_add_check("deterministic_action_unique_ratio_mean", model_det_action_ratio.mean(), _status_high_good(model_det_action_ratio.mean(), 0.95, 0.80), "Action diversity close to 1.0 means the policy is not repeating the exact same weight vector every day.")
_add_check("deterministic_argmax_alpha_uniques_min", model_det_argmax.min(), _status_high_good(model_det_argmax.min(), 6.0, 4.0), "The top-conviction slot should rotate across a reasonable share of assets.")
_add_check("deterministic_alpha_le1_fraction_mean", model_det_alpha_le1.mean(), _status_low_good(model_det_alpha_le1.mean(), 0.70, 0.85), "Too many alpha<=1 steps can indicate compressed conviction.")

for method_name in ["stratified", "monte_carlo"]:
    ss = stoch_summary_combined_df[stoch_summary_combined_df["method"] == method_name]
    if ss.empty:
        continue
    mean_sharpe = pd.to_numeric(ss["sharpe_mean"], errors="coerce").mean()
    pos_rate = pd.to_numeric(ss["positive_sharpe_rate_pct"], errors="coerce").mean()
    p10_ann = pd.to_numeric(ss["ann_ret_p10_pct"], errors="coerce").mean()
    cvar10_ann = pd.to_numeric(ss["ann_ret_cvar10_pct"], errors="coerce").mean()
    mean_mdd = pd.to_numeric(ss["mdd_mean_pct"], errors="coerce").mean()
    mean_gap = pd.to_numeric(generalizability_by_horizon_df.loc[generalizability_by_horizon_df["method"] == method_name, "det_vs_sto_sharpe_gap"], errors="coerce").mean()
    _add_check(f"{method_name}_mean_sharpe", mean_sharpe, _status_high_good(mean_sharpe, 0.50, 0.25), "Average stochastic Sharpe should remain positive and useful.")
    _add_check(f"{method_name}_positive_sharpe_rate_pct", pos_rate, _status_high_good(pos_rate, 70.0, 55.0), "A robust model should produce positive Sharpe in most random windows.")
    _add_check(f"{method_name}_ann_return_p10_pct", p10_ann, _status_high_good(p10_ann, 0.0, -2.0), "10th percentile annualized return should not be deeply negative.")
    _add_check(f"{method_name}_ann_return_cvar10_pct", cvar10_ann, _status_high_good(cvar10_ann, 0.0, -4.0), "Left-tail annualized return should stay acceptable.")
    _add_check(f"{method_name}_mean_mdd_pct", mean_mdd, _status_low_good(mean_mdd, 25.0, 30.0), "Average stochastic drawdown should stay within project risk limits.")
    _add_check(f"{method_name}_det_to_stoch_sharpe_gap", mean_gap, _status_low_good(mean_gap, 0.45, 0.75), "Large deterministic-vs-stochastic gaps indicate overfit or seed sensitivity.")

checks_df = pd.DataFrame(checks)
overall_verdict = _overall_generalizability_verdict(checks_df)
status_counts = checks_df["status"].value_counts(dropna=False).to_dict()

generalizability_report = {
    "checkpoint_tag": CHECKPOINT_TAG,
    "artifact_tag": ARTIFACT_TAG,
    "overall_verdict": overall_verdict,
    "status_counts": status_counts,
    "deterministic_horizons_tested": int(len(model_det_df)),
    "stochastic_methods_tested": sorted(stoch_summary_combined_df["method"].dropna().unique().tolist()) if not stoch_summary_combined_df.empty else [],
    "checks": checks_df.to_dict(orient="records"),
}

generalizability_checks_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_generalizability_checks.csv"
generalizability_report_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_generalizability_report.json"
checks_df.to_csv(generalizability_checks_path, index=False)
_safe_write_json(generalizability_report_path, generalizability_report)

print("=" * 80)
print("GENERALIZABILITY CHECK")
print("=" * 80)
print("Overall verdict:", overall_verdict)
print("Artifacts:")
print("  by horizon:", generalizability_by_horizon_path)
print("  checks:", generalizability_checks_path)
print("  report:", generalizability_report_path)
display(generalizability_by_horizon_df)
display(checks_df)


## 13) Generalizability Visuals


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) Deterministic vs stochastic Sharpe by horizon and method.
ax = axes[0, 0]
if not generalizability_by_horizon_df.empty:
    plot_df = generalizability_by_horizon_df.copy()
    for method_name, color in [("stratified", COLORS["stratified"]), ("monte_carlo", COLORS["monte_carlo"])]:
        ss = plot_df[plot_df["method"] == method_name].sort_values("horizon", key=lambda s: s.map(_horizon_sort_key))
        if not ss.empty:
            ax.plot(ss["horizon"], ss["sto_sharpe_mean"], marker="o", linewidth=2, color=color, label=f"{method_name} stochastic")
    det_sorted = model_det_df.sort_values("horizon", key=lambda s: s.map(_horizon_sort_key))
    ax.plot(det_sorted["horizon"], det_sorted["sharpe_ratio"], marker="s", linewidth=2, color=COLORS["model"], label="deterministic")
ax.set_title("Sharpe Stability Across Horizons")
ax.set_ylabel("Sharpe")
ax.grid(alpha=0.25)
ax.legend()

# 2) Deterministic vs stochastic annualized return by horizon.
ax = axes[0, 1]
if not generalizability_by_horizon_df.empty:
    plot_df = generalizability_by_horizon_df.copy()
    for method_name, color in [("stratified", COLORS["stratified"]), ("monte_carlo", COLORS["monte_carlo"])]:
        ss = plot_df[plot_df["method"] == method_name].sort_values("horizon", key=lambda s: s.map(_horizon_sort_key))
        if not ss.empty:
            ax.plot(ss["horizon"], ss["sto_ann_ret_mean_pct"], marker="o", linewidth=2, color=color, label=f"{method_name} stochastic")
    det_sorted = model_det_df.sort_values("horizon", key=lambda s: s.map(_horizon_sort_key))
    ax.plot(det_sorted["horizon"], det_sorted["ann_return_pct"], marker="s", linewidth=2, color=COLORS["model"], label="deterministic")
ax.set_title("Annualized Return Stability Across Horizons")
ax.set_ylabel("Annualized Return (%)")
ax.grid(alpha=0.25)
ax.legend()

# 3) Generalizability check status counts.
ax = axes[1, 0]
status_order = ["pass", "watch", "fail", "missing"]
status_counts_plot = pd.Series(status_counts).reindex(status_order).fillna(0)
bar_colors = {"pass": "#2ca02c", "watch": "#ffbf00", "fail": "#d62728", "missing": "#7f7f7f"}
ax.bar(status_counts_plot.index, status_counts_plot.values, color=[bar_colors[s] for s in status_counts_plot.index])
ax.set_title(f"Generalizability Verdict: {overall_verdict}")
ax.set_ylabel("Check Count")
ax.grid(axis="y", alpha=0.25)

# 4) Heatmap of det-vs-stochastic Sharpe gap.
ax = axes[1, 1]
heat_df = generalizability_by_horizon_df.pivot_table(index="method", columns="horizon", values="det_vs_sto_sharpe_gap", aggfunc="first")
heat_df = heat_df.reindex(index=["stratified", "monte_carlo"], columns=HORIZON_ORDER)
if not heat_df.empty:
    im = ax.imshow(heat_df.fillna(0.0).values, aspect="auto", cmap="RdYlGn_r")
    ax.set_xticks(range(len(heat_df.columns)))
    ax.set_xticklabels(list(heat_df.columns))
    ax.set_yticks(range(len(heat_df.index)))
    ax.set_yticklabels(list(heat_df.index))
    ax.set_title("Deterministic - Stochastic Sharpe Gap")
    for i in range(len(heat_df.index)):
        for j in range(len(heat_df.columns)):
            val = heat_df.iloc[i, j]
            txt = "n/a" if pd.isna(val) else f"{val:.2f}"
            ax.text(j, i, txt, ha="center", va="center", color="black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
else:
    ax.set_title("Deterministic - Stochastic Sharpe Gap (n/a)")
    ax.axis("off")

fig.suptitle(f"Generalizability Dashboard — {CHECKPOINT_TAG}", fontsize=14, fontweight="bold")
plt.tight_layout()
generalizability_dashboard_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_generalizability_dashboard.png"
fig.savefig(generalizability_dashboard_path, dpi=150, bbox_inches="tight")
plt.show()

print("[OK] Saved generalizability dashboard:", generalizability_dashboard_path)


In [ ]:
import matplotlib.pyplot as plt


def _strategy_metric_frame(df_long, metric):
    out = (
        df_long.pivot_table(index="horizon", columns="strategy", values=metric, aggfunc="first")
        .reindex(HORIZON_ORDER)
    )
    return out


det_strategy_plot_df = det_benchmark_long_df.copy()
det_strategy_plot_df["strategy"] = det_strategy_plot_df["strategy"].replace({"model": "Model"})
det_strategy_plot_df = det_strategy_plot_df[det_strategy_plot_df["strategy"].isin(["Model", "SPY", "EqualWeight"])].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
metrics = [
    ("ann_return_pct", "Annualized Return (%)"),
    ("sharpe_ratio", "Sharpe Ratio"),
    ("max_dd_pct", "Max Drawdown (%)"),
    ("turnover_pct", "Turnover (%)"),
]
for ax, (metric, title) in zip(axes.ravel(), metrics):
    plot_df = _strategy_metric_frame(det_strategy_plot_df.rename(columns={"strategy": "strategy"}), metric)
    if plot_df.empty:
        ax.set_title(f"{title} (n/a)")
        continue
    x = np.arange(len(plot_df.index))
    width = 0.25
    for i, strategy_name in enumerate(plot_df.columns):
        ax.bar(x + i * width, plot_df[strategy_name].values, width=width, label=strategy_name, color=COLORS.get(strategy_name, None))
    ax.set_xticks(x + width)
    ax.set_xticklabels(plot_df.index)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    if metric == "max_dd_pct":
        ax.invert_yaxis()
axes[0, 0].legend()
fig.suptitle(f"Deterministic Horizon Comparison vs Benchmarks — {CHECKPOINT_TAG}", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_vs_benchmarks.png", dpi=150, bbox_inches="tight")
plt.show()


fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, horizon_label in zip(axes.ravel(), HORIZON_ORDER):
    subset = stoch_runs_combined_df[stoch_runs_combined_df["horizon"] == horizon_label].copy()
    if subset.empty:
        ax.set_title(f"{horizon_label} (n/a)")
        continue
    for method_name in ["stratified", "monte_carlo"]:
        ss = subset[subset["method"] == method_name]
        ax.scatter(
            pd.to_numeric(ss["max_dd_pct"], errors="coerce"),
            pd.to_numeric(ss["ann_return_pct"], errors="coerce"),
            alpha=0.65,
            label=method_name,
            color=COLORS[method_name],
        )
    ax.set_title(horizon_label)
    ax.set_xlabel("Max Drawdown (%)")
    ax.set_ylabel("Annualized Return (%)")
    ax.grid(alpha=0.25)
axes[0, 0].legend()
fig.suptitle("Stochastic Risk-Return Frontier by Horizon", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"{ARTIFACT_TAG}_stochastic_frontier.png", dpi=150, bbox_inches="tight")
plt.show()


fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, horizon_label in zip(axes.ravel(), HORIZON_ORDER):
    subset = stoch_runs_combined_df[stoch_runs_combined_df["horizon"] == horizon_label].copy()
    if subset.empty:
        ax.set_title(f"{horizon_label} (n/a)")
        continue
    strat_vals = pd.to_numeric(subset.loc[subset["method"] == "stratified", "sharpe_ratio"], errors="coerce").dropna().values
    mc_vals = pd.to_numeric(subset.loc[subset["method"] == "monte_carlo", "sharpe_ratio"], errors="coerce").dropna().values
    data = [strat_vals, mc_vals]
    labels = ["stratified", "monte_carlo"]
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.55)
    for patch, color in zip(bp["boxes"], [COLORS["stratified"], COLORS["monte_carlo"]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.axhline(y=0.0, color="red", linestyle="--", alpha=0.5)
    ax.set_title(f"{horizon_label} Sharpe")
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("Stochastic Sharpe Distribution by Sampling Method", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"{ARTIFACT_TAG}_stochastic_sharpe_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
avg_weight_rows = []
final_weight_rows = []
for horizon_label, df in det_daily_data.items():
    weight_cols = [c for c in df.columns if c.startswith("w_")]
    if not weight_cols:
        continue
    for col in weight_cols:
        ticker = col.replace("w_", "")
        avg_weight_rows.append({
            "horizon": horizon_label,
            "ticker": ticker,
            "avg_weight": float(pd.to_numeric(df[col], errors="coerce").mean()),
        })
        final_weight_rows.append({
            "horizon": horizon_label,
            "ticker": ticker,
            "final_weight": float(pd.to_numeric(df[col], errors="coerce").iloc[-1]),
        })

det_weight_profile_df = pd.DataFrame(avg_weight_rows)
det_final_weight_df = pd.DataFrame(final_weight_rows)
det_asset_profile_df = det_asset_vs_spy_df.merge(det_weight_profile_df, on=["horizon", "ticker"], how="left")
det_asset_profile_df = det_asset_profile_df.merge(det_final_weight_df, on=["horizon", "ticker"], how="left")
det_asset_profile_df.to_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_asset_profile.csv", index=False)

weight_heatmap = det_asset_profile_df.pivot_table(index="horizon", columns="ticker", values="avg_weight", aggfunc="first").reindex(HORIZON_ORDER)
excess_heatmap = det_asset_profile_df.pivot_table(index="horizon", columns="ticker", values="asset_excess_vs_spy_pct", aggfunc="first").reindex(HORIZON_ORDER)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
if not weight_heatmap.empty:
    im1 = axes[0].imshow(weight_heatmap.fillna(0.0).values, aspect="auto", cmap="Blues")
    axes[0].set_yticks(np.arange(len(weight_heatmap.index)))
    axes[0].set_yticklabels(weight_heatmap.index)
    axes[0].set_xticks(np.arange(len(weight_heatmap.columns)))
    axes[0].set_xticklabels(weight_heatmap.columns, rotation=45, ha="right")
    axes[0].set_title("Average Deterministic Weights by Horizon")
    fig.colorbar(im1, ax=axes[0], fraction=0.025, pad=0.02)
if not excess_heatmap.empty:
    im2 = axes[1].imshow(excess_heatmap.fillna(0.0).values, aspect="auto", cmap="RdYlGn")
    axes[1].set_yticks(np.arange(len(excess_heatmap.index)))
    axes[1].set_yticklabels(excess_heatmap.index)
    axes[1].set_xticks(np.arange(len(excess_heatmap.columns)))
    axes[1].set_xticklabels(excess_heatmap.columns, rotation=45, ha="right")
    axes[1].set_title("Asset Total Return Excess vs SPY by Horizon (%)")
    fig.colorbar(im2, ax=axes[1], fraction=0.025, pad=0.02)
fig.suptitle("Deterministic Asset Allocation vs Realized Asset Opportunity Set", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_asset_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, horizon_label in zip(axes.ravel(), HORIZON_ORDER):
    subset = det_asset_profile_df[det_asset_profile_df["horizon"] == horizon_label].copy()
    if subset.empty:
        ax.set_title(f"{horizon_label} (n/a)")
        continue
    subset = subset.sort_values("avg_weight", ascending=False)
    x = np.arange(len(subset))
    ax.bar(x, subset["avg_weight"].values, color="#1f77b4", alpha=0.7, label="Avg model weight")
    ax2 = ax.twinx()
    ax2.plot(x, subset["asset_excess_vs_spy_pct"].values, color="#ff7f0e", marker="o", linewidth=1.5, label="Asset excess vs SPY")
    ax.axhline(y=0.0, color="black", alpha=0.2)
    ax2.axhline(y=0.0, color="red", linestyle="--", alpha=0.4)
    ax.set_xticks(x)
    ax.set_xticklabels(subset["ticker"].tolist(), rotation=45, ha="right")
    ax.set_title(horizon_label)
    ax.set_ylabel("Average Weight")
    ax2.set_ylabel("Excess Return vs SPY (%)")
fig.suptitle("Deterministic Weights vs Asset Performance Relative to SPY", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_weights_vs_spy.png", dpi=150, bbox_inches="tight")
plt.show()


## 12) Full OOS Single Pass — Optional


In [ ]:
print(f"Running full OOS deterministic evaluation from {EVAL_FORCE_TEST_START_DATE} to end of test data...")
full_oos_eval = evaluate_experiment6_checkpoint(
    experiment6=experiment6,
    phase1_data=eval_phase1_data,
    config=eval_config,
    random_seed=EVAL_RANDOM_SEED,
    checkpoint_path_override=str(hw_dir / CHECKPOINT_TAG),
    deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
    stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
    num_eval_runs=0,
    stochastic_episode_length_limit=9999,
    save_eval_logs=EVAL_SAVE_LOGS,
    save_eval_artifacts=EVAL_SAVE_ARTIFACTS,
)

dm = full_oos_eval.deterministic_metrics or {}
pv = np.asarray(full_oos_eval.deterministic_portfolio) if full_oos_eval.deterministic_portfolio is not None else np.empty((0,))
dw = np.asarray(full_oos_eval.deterministic_weights) if full_oos_eval.deterministic_weights is not None else np.empty((0, 0))
da = np.asarray(full_oos_eval.deterministic_alphas) if full_oos_eval.deterministic_alphas is not None else np.empty((0, 0))

if pv.size == 0:
    raise RuntimeError("deterministic_portfolio is empty")

test_df_eval = getattr(full_oos_eval.env_test_deterministic, "processed_data", pd.DataFrame()).copy()
if isinstance(test_df_eval, pd.DataFrame) and "Date" in test_df_eval.columns:
    full_dates = (
        pd.to_datetime(test_df_eval["Date"]).dropna().drop_duplicates().sort_values().reset_index(drop=True)
    )
else:
    full_dates = pd.Series(pd.NaT, index=np.arange(len(pv)))

n = len(pv)
if dw.ndim == 2 and dw.shape[0] > 0:
    n = min(n, dw.shape[0])
if da.ndim == 2 and da.shape[0] > 0:
    n = min(n, da.shape[0])
if len(full_dates) > 0:
    n = min(n, len(full_dates))

full_daily_df = pd.DataFrame({
    "step": np.arange(n),
    "date": pd.to_datetime(full_dates.iloc[:n].values if len(full_dates) >= n else pd.NaT, errors="coerce"),
    "portfolio_value": pv[:n],
})
full_daily_df["daily_return"] = full_daily_df["portfolio_value"].pct_change().fillna(0.0)
full_daily_df["cumulative_return"] = full_daily_df["portfolio_value"] / full_daily_df["portfolio_value"].iloc[0] - 1.0
running_max = full_daily_df["portfolio_value"].cummax()
full_daily_df["drawdown"] = full_daily_df["portfolio_value"] / running_max.replace(0, np.nan) - 1.0

if dw.ndim == 2 and dw.shape[0] >= n:
    for i, ticker in enumerate(EVAL_ASSET_UNIVERSE):
        if i < dw.shape[1]:
            full_daily_df[f"w_{ticker}"] = dw[:n, i]
    if dw.shape[1] > len(EVAL_ASSET_UNIVERSE):
        full_daily_df["w_cash"] = dw[:n, -1]

if da.ndim == 2 and da.shape[0] >= n:
    for i, ticker in enumerate(EVAL_ASSET_UNIVERSE):
        if i < da.shape[1]:
            full_daily_df[f"alpha_{ticker}"] = da[:n, i]
    if da.shape[1] > len(EVAL_ASSET_UNIVERSE):
        full_daily_df["alpha_cash"] = da[:n, -1]

print(
    f"[OK] Full OOS | Final=${full_daily_df['portfolio_value'].iloc[-1]:,.2f} | "
    f"Sharpe={_fmt(dm.get('sharpe_ratio', np.nan), 4)} | "
    f"Return={_fmt((dm.get('total_return', np.nan) * 100.0) if dm.get('total_return') is not None else np.nan, 2, '%')} | "
    f"MDD={_fmt((dm.get('max_drawdown_abs', np.nan) * 100.0) if dm.get('max_drawdown_abs') is not None else np.nan, 2, '%')}"
)
display(full_daily_df.head())


## 13) Save Everything + Drive Backup


In [ ]:
full_csv_path = OUTPUT_DIR / f"{ARTIFACT_TAG}_full_oos_daily.csv"
full_daily_df.to_csv(full_csv_path, index=False)

export_summary = {
    "eval_config_snapshot_path": str(eval_config_snapshot_path),
    "hyperparameter_snapshot_path": str(hyperparameter_snapshot_path),
    "eval_healthcheck_path": str(eval_healthcheck_path),
    "generalizability_by_horizon_path": str(generalizability_by_horizon_path),
    "generalizability_checks_path": str(generalizability_checks_path),
    "generalizability_report_path": str(generalizability_report_path),
    "generalizability_dashboard_path": str(generalizability_dashboard_path),
    "det_results_df": tuple(det_results_df.shape),
    "det_benchmark_long_df": tuple(det_benchmark_long_df.shape),
    "det_comparison_df": tuple(det_comparison_df.shape),
    "det_asset_profile_df": tuple(det_asset_profile_df.shape),
    "stratified_stoch_df": tuple(stratified_stoch_df.shape),
    "mc_stoch_df": tuple(mc_stoch_df.shape),
    "stoch_summary_combined_df": tuple(stoch_summary_combined_df.shape),
    "generalizability_by_horizon_df": tuple(generalizability_by_horizon_df.shape),
    "generalizability_checks_df": tuple(checks_df.shape),
    "full_daily_df": tuple(full_daily_df.shape),
    "output_dir": str(OUTPUT_DIR),
}

with open(OUTPUT_DIR / f"{ARTIFACT_TAG}_export_summary.json", "w", encoding="utf-8") as f:
    json.dump(export_summary, f, indent=2, default=str)

print("=" * 80)
print("DATA EXPORT SUMMARY")
print("=" * 80)
for k, v in export_summary.items():
    print(f"{k}: {v}")
print("=" * 80)


In [ ]:
import zipfile

if Path("/content/drive/MyDrive").exists():
    backup_zip = Path(f"/content/drive/MyDrive/robustness_{ARTIFACT_TAG}_{RUN_ID}.zip")
else:
    backup_zip = OUTPUT_DIR.parent / f"robustness_{ARTIFACT_TAG}_{RUN_ID}.zip"

with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in OUTPUT_DIR.rglob("*"):
        if fp.is_file():
            zf.write(fp, fp.relative_to(OUTPUT_DIR))

print(f"[OK] Backup zip: {backup_zip} ({len(list(OUTPUT_DIR.rglob('*')))} files from {OUTPUT_DIR})")


## 14) Reload Saved Artifacts


In [ ]:
# Uncomment to reload key outputs:
# det_results_df = pd.read_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_summary.csv")
# det_benchmark_long_df = pd.read_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_benchmarks_long.csv")
# det_comparison_df = pd.read_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_comparison.csv")
# det_asset_profile_df = pd.read_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_deterministic_asset_profile.csv")
# stratified_stoch_df = pd.read_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_stratified_stochastic_all.csv")
# mc_stoch_df = pd.read_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_monte_carlo_stochastic_all.csv")
# stoch_summary_combined_df = pd.read_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_stochastic_method_summary.csv")
# full_daily_df = pd.read_csv(OUTPUT_DIR / f"{ARTIFACT_TAG}_full_oos_daily.csv", parse_dates=["date"])
# print(f"[OK] Reloaded artifacts from {OUTPUT_DIR}")
